# Прогнозный выбор целых подграфов

Все данные синтетические. Работа выполняется в отдельной ветке; эталон v2 не изменяется.

Найдём корень проекта. В Colab при отсутствии файлов клонируется только рабочая ветка; локально используются уже полученные исходники.

In [ ]:
from pathlib import Path
import os, sys
root = Path.cwd()
while root != root.parent and not (root/'efc').exists():
    root = root.parent
if not (root/'efc').exists():
    import subprocess
    target = Path('/content/Forecast_Edge_Fog_Cloud')
    subprocess.run(['git','clone','--branch','night/fefc-v3','https://github.com/annfridreksen/Forecast_Edge_Fog_Cloud.git',str(target)],check=True)
    root=target
os.chdir(root)
sys.path.insert(0,str(root))
print('Рабочая папка:',root)


Что исследуем: работоспособность неизменённого v2 и подключение команд от демонстрационной сети. Облако не включается в кандидаты; это граница постановки, мотивированная прежним результатом НСКФ, не универсальное свойство облака.

In [ ]:
from efc.placement.study import controls,neural_placement
from efc.placement.views import placement_plots
import json,pandas as pd
control,plans,audits=controls()
display(control)

Посмотрим порядок реально проверенных назначений и оставшуюся границу. Ограниченный поиск не объявляется доказательством невозможности, а число проверок — временем исполнения.

In [ ]:
print(json.dumps(audits['minimal_P'],ensure_ascii=False,indent=2,default=str))
assert plans['minimal_P']['plan']['makespan']=='1/125'
assert plans['minimal_B']['plan']['makespan']=='1/250'

Второй расчёт использует предсказания сети из тетради02. Если она ещё не запускалась, сначала получаем её результаты. Очереди Fog0/Fog1 взяты из временной точки той же среды; P и B получают одинаковый снимок.

In [ ]:
if not Path('results/demo_network.json').exists():
    from efc.forecast.study import forecast_study
    forecast_study()
linked,payload=neural_placement()
display(linked)
print('Прогнозы:',payload['predictions'])
print('Ранги:',payload['ranks'])
print(payload['scope'])

Граф и диаграммы Ганта строятся из проверенных планов, не рисуются вручную по ожидаемому ответу. Синий/оранжевый цвет обозначает устройство целого подграфа.

In [ ]:
placement_plots(plans,payload)

Проверка: известные контрольные числа должны совпасть точно. Результат P/B сквозного примера сохраняется при любом знаке разности; преимущество по одному случаю не устанавливается.

In [ ]:
assert list(control.time_ms.iloc[:3])==[8,4,6]
assert control.status.iloc[3]=='NO_PLAN_IN_CLASS'
assert len(linked)==2
assert all(x['violations']==[] for x in payload['replay'].values())
display(linked.rename(columns={'planned_ms':'план: статья = модель','actual_ms':'исполнение: статья = модель'}))
print('OK')